In [1]:
import nltk
nltk.download('punkt')      # for tokenization
nltk.download('averaged_perceptron_tagger')  # for POS tagging


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.


True

In [ ]:
import re
import nltk
from nltk import pos_tag, word_tokenize
from nltk.tree import Tree
from nltk.chunk import RegexpParser


# Emotion to Emoji Mapping
emoji_map = {
    'happy': '😊',
    'sad': '😔',
    'disgust': '🤢',
    'fear': '😨',
    'angry': '😠'
}

# Emotion Lexicon with weights
emotion_lexicon = {
    'happy': {'love': 2, 'like': 1, 'joy': 2, 'delicious': 1, 'great': 1, 'happy': 2, 'excited': 2},
    'sad': {'sad': 2, 'failed': 1, 'crying': 2, 'regret': 1, 'unhappy': 2, 'disappointed': 1},
    'disgust': {'hate': 2, 'disgusting': 3, 'gross': 2, 'yuck': 1, 'nasty': 2},
    'fear': {'scared': 2, 'afraid': 1, 'terrified': 3, 'nightmare': 2, 'horror': 2, 'panic': 1},
    'angry': {'angry': 2, 'furious': 3, 'mad': 2, 'annoyed': 1, 'rage': 3}
}

negations = {"not", "no", "never", "don't", "didn't", "isn't", "wasn't", "aren't", "can't", "won't"}

# Sentences to test
sentences = [
    "I love pizza",
    "I hate this food",
    "She got me a burger",
    "I don't like burgers",
    "This is disgusting",
    "I'm scared of the dark",
    "He failed his test again",
    "Wow, that went great!",
    "What a nightmare",
    "I’m not happy with the results"
]

# Preprocessing
def preprocess(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r"[^\w\s']", "", sentence)
    return sentence.split()

# Use parse tree to extract relevant chunks (noun/adjective phrases)
def get_phrases(sentence):
    tokens = word_tokenize(sentence)
    tagged = pos_tag(tokens)

    grammar = r"""
        NP: {<DT>?<JJ.*>*<NN.*>+}       # Noun phrases
        ADJP: {<RB.?>*<JJ>}             # Adjective phrases
    """
    parser = RegexpParser(grammar)
    tree = parser.parse(tagged)

    key_chunks = []
    for subtree in tree:
        if isinstance(subtree, Tree):
            phrase = " ".join(word for word, tag in subtree.leaves())
            key_chunks.append(phrase.lower())
    return key_chunks

# Scoring with negation, weights, and phrase importance
def score_sentence(sentence):
    words = preprocess(sentence)
    phrases = get_phrases(sentence)
    score = {emotion: 0 for emotion in emoji_map}

    for i, word in enumerate(words):
        is_negated = False
        for offset in range(1, 4):
            if i - offset >= 0 and words[i - offset] in negations:
                is_negated = True
                break

        for emotion, keywords in emotion_lexicon.items():
            if word in keywords:
                base_weight = keywords[word]
                if any(word in phrase for phrase in phrases):
                    base_weight += 1  # boost for being in a noun/adj phrase
                if is_negated:
                    if emotion == 'happy':
                        score['sad'] += base_weight
                    elif emotion == 'sad':
                        score['happy'] += base_weight
                    elif emotion == 'disgust':
                        score['happy'] += base_weight
                    else:
                        score[emotion] -= base_weight
                else:
                    score[emotion] += base_weight
    return score

# Predict using max score
def predict_emoji(sentence):
    scores = score_sentence(sentence)
    best_emotion = max(scores, key=scores.get)
    return emoji_map.get(best_emotion, '🤔')

# Run it
for s in sentences:
    print(f"{s} → {predict_emoji(s)}")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


I love pizza → 😊
I hate this food → 🤢
She got me a burger → 😊
I don't like burgers → 😔
This is disgusting → 🤢
I'm scared of the dark → 😨
He failed his test again → 😔
Wow, that went great! → 😊
What a nightmare → 😨
I’m not happy with the results → 😔
